<a href="https://colab.research.google.com/github/enedenao/tp_labo/blob/nao/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP Grupal — Laboratorio de Métodos Cuantitativos (FCE UBA, 2C 2026)

**Integrantes:** Bautista Cortés · Gabriel Goldadler · José Luis Joannás · Naomi Asato · @dagut-rrd

**Repositorio:** https://github.com/enedenao/tp_labo

**Pregunta de investigación (versión actualizada):**

¿Qué personajes concentran el diálogo en Los Simpson, y existe una asociación entre el protagonismo de los personajes principales y la valoración de los episodios en IMDb que se mantenga a lo largo de las décadas?

## 0. Configuración del entorno

Importamos las bibliotecas que vamos a usar en todo el análisis:

- **pandas**: carga y manipulación del dataset (DataFrames, `groupby`, filtros).
- **numpy**: operaciones numéricas auxiliares.
- **matplotlib** y **seaborn**: gráficos. Seaborn se apoya en matplotlib y simplifica gráficos estadísticos (boxplots, barras agrupadas).
- **os**: crear la carpeta donde se guardan los gráficos exportados para el informe.

In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("figuras", exist_ok=True)

## 1. Carga del dataset

**Dataset:** The Simpsons Data recopilación de Todd Schneider

**Fuente:** publicado en Kaggle

**Página del recurso:** https://github.com/sujanjoejacob/Text-mining-with-Simpsons-Data


Cargamos el CSV/ZIP directamente desde el repo, para que el notebook sea reproducible sin depender de archivos locales.

In [58]:
URL_EPISODIOS = "https://raw.githubusercontent.com/enedenao/tp_labo/main/data/simpsons_episodes.csv"
URL_DIALOGOS = "https://raw.githubusercontent.com/enedenao/tp_labo/main/data/simpsons_script_lines.csv.zip"

episodios = pd.read_csv(URL_EPISODIOS)
dialogos = pd.read_csv(URL_DIALOGOS, on_bad_lines="skip", low_memory=False)

print(f"Episodios: {episodios.shape[0]:,} filas | {episodios.shape[1]} columnas")
print(f"Dialogos:   {dialogos.shape[0]:,} filas | {dialogos.shape[1]} columnas")

Episodios: 600 filas | 13 columnas
Dialogos:   158,248 filas | 13 columnas


**Decisión de limpieza en la carga:** algunas filas del archivo de diálogos están mal armadas: tienen comas de más dentro del texto, y pandas no puede separar bien las columnas. Por defecto, eso hace que la carga falle. Con `on_bad_lines="skip"` se descartan esas filas: son 23 sobre 158.271 (menos del 0,02%), por lo que no afectan el análisis.

> **Asistencia de IA:** Claude (Claude Project "tp labo"). Se le compartió el CSV de diálogos y detectó las filas mal formadas; propuso `on_bad_lines="skip"`.

## 3.1 Exploración inicial

### a) Primeras y últimas filas

Mostramos el inicio y el final de cada tabla para ver cómo vienen los datos y detectar problemas a simple vista.

In [45]:
episodios.head()

,id,title,original_air_date,production_code,season,number_in_season,number_in_series,us_viewers_in_millions,views,imdb_rating,imdb_votes,image_url,video_url
0,10,Homer's Night Out,1990-03-25,7G10,1,10,10,30.3,50816.0,7.4,1511.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/275197507879
1,12,Krusty Gets Busted,1990-04-29,7G12,1,12,12,30.4,62561.0,8.3,1716.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/288019523914
2,14,"Bart Gets an ""F""",1990-10-11,7F03,2,1,14,33.6,59575.0,8.2,1638.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/260539459671
3,17,Two Cars in Every Garage and Three Eyes on Eve...,1990-11-01,7F01,2,4,17,26.1,64959.0,8.1,1457.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/260537411822
4,19,Dead Putting Society,1990-11-15,7F08,2,6,19,25.4,50691.0,8.0,1366.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/260539459670


In [46]:
episodios.tail()

,id,title,original_air_date,production_code,season,number_in_season,number_in_series,us_viewers_in_millions,views,imdb_rating,imdb_votes,image_url,video_url
595,500,At Long Last Leave,2012-02-19,PABF07,23,14,500,5.77,45226.0,7.0,580.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/230541891832
596,502,How I Wet Your Mother,2012-03-11,PABF08,23,16,502,4.97,44818.0,7.3,541.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/230597187582
597,511,Adventures in Baby-Getting,2012-11-04,PABF18,24,3,511,5.65,39687.0,6.9,460.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/221688899646
598,542,Diggs,2014-03-09,SABF08,25,12,542,2.69,39292.0,6.4,473.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/310529091639
599,548,Days of Future Future,2014-04-13,SABF13,25,18,548,3.64,55742.0,7.0,504.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/310257731892


In [47]:
dialogos.head()

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,true,464,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31.0
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,true,9,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3.0
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,true,464,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22.0
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,true,9,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5.0
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,true,40,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33.0


In [48]:
dialogos.tail()

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
158243,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,true,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2.0
158244,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,true,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10.0
158245,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,true,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1.0
158246,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,true,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6.0
158247,9548,32,208,"JANEY: No, that means she was faking it.",845000,true,4,3.0,JANEY,Springfield Elementary School,"No, that means she was faking it.",no that means she was faking it,7.0


### b) Estructura: dimensiones, columnas, tipos de dato y cobertura

`.dtypes` muestra el tipo de dato de cada columna. Una columna que debería ser numérica pero aparece como texto (`object`) suele indicar datos sucios.

In [49]:
print(episodios.dtypes)

id                          int64
title                      object
original_air_date          object
production_code            object
season                      int64
number_in_season            int64
number_in_series            int64
us_viewers_in_millions    float64
views                     float64
imdb_rating               float64
imdb_votes                float64
image_url                  object
video_url                  object
dtype: object


In [50]:
print(dialogos.dtypes)

id                      int64
episode_id              int64
number                  int64
raw_text               object
timestamp_in_ms        object
speaking_line          object
character_id           object
location_id           float64
raw_character_text     object
raw_location_text      object
spoken_words           object
normalized_text        object
word_count            float64
dtype: object


In [51]:
print("Fecha de emisión: de", episodios["original_air_date"].min(), "a", episodios["original_air_date"].max())
print("Temporadas:", episodios["season"].nunique())
print("Episodios en la tabla de episodios:", episodios["id"].nunique())                    #nunique() cuenta cuántos valores distintos hay en una columna.
print("Episodios con líneas de diálogo:", dialogos["episode_id"].nunique())

Fecha de emisión: de 1989-12-17 a 2016-10-16
Temporadas: 28
Episodios en la tabla de episodios: 600
Episodios con líneas de diálogo: 564


Hay 36 episodios sin dialogo

### c) Calidad de los datos: nulos, duplicados y estadísticas descriptivas

Contamos valores faltantes y filas repetidas, y resumimos las columnas numéricas. Revisamos además los valores de `speaking_line`, que indica si una línea es diálogo o una acotación del guion.

In [52]:
print(episodios.isnull().sum()) #tenemos 3 episodios (imdb_rating) sin datos que representa el 0.5%
print("Filas duplicadas:", episodios.duplicated().sum()) #no hay duplicados

id                        0
title                     0
original_air_date         0
production_code           0
season                    0
number_in_season          0
number_in_series          0
us_viewers_in_millions    6
views                     4
imdb_rating               3
imdb_votes                3
image_url                 4
video_url                 4
dtype: int64
Filas duplicadas: 0


In [53]:
print(dialogos.isnull().sum()) #el exceso de nulos corresponden a acotaciones del gruion
print("Filas duplicadas:", dialogos.duplicated().sum()) #no tenemos filas duplicadas

id                        0
episode_id                0
number                    0
raw_text                  0
timestamp_in_ms           0
speaking_line             0
character_id          17521
location_id             407
raw_character_text    17522
raw_location_text       408
spoken_words          26159
normalized_text       26184
word_count            26159
dtype: int64
Filas duplicadas: 0


In [54]:
episodios[["imdb_rating", "us_viewers_in_millions", "imdb_votes"]].describe()

,imdb_rating,us_viewers_in_millions,imdb_votes
count,597.000000,594.000000,597.000000
mean,7.386097,11.842795,832.365159
std,0.732439,6.414218,396.117937
min,4.500000,2.320000,104.000000
25%,6.900000,7.055000,560.000000
50%,7.300000,10.300000,697.000000
75%,8.000000,15.250000,1095.000000
max,9.200000,33.600000,3734.000000


**Rating**:

Promedio 7,4, mediana 7,3, desvío 0,73.

El 50% de los episodios está entre 6,9 y 8,0 (cuartiles 25% y 75%).

El mínimo, 4,5, está muy por debajo del resto: es un episodio atípico.

Que el rango sea estrecho importa para la pregunta: las diferencias de rating entre episodios son chicas, así que cualquier asociación con el protagonismo probablemente sea moderada.

**Audiencia**:

Promedio 11,8 millones, pero mediana 10,3. Cuando el promedio es mayor que la mediana, hay valores altos que “tiran” el promedio para arriba. El rango va de 2,3 a 33,6 millones, lo que sugiere una caída importante de la audiencia a lo largo del tiempo, a verificar en el análisis de gráfico.

In [55]:
dialogos["word_count"].describe()
#Las 132.089 líneas con palabras registradas tienen en promedio casi 10 palabras, con una mediana de 8: predominan los diálogos breves.

,word_count
count,132089.000000
mean,9.887175
std,8.055425
min,0.000000
25%,4.000000
50%,8.000000
75%,13.000000
max,122.000000


In [56]:
dialogos["speaking_line"].value_counts(dropna=False) #132.089 líneas habladas y 26.158 acotaciones.
#hay una fila que es dialogo y quedo en columna equivocada

,count
speaking_line,
true,132089
false,26158
Guess what. I also play Frankenstein!,1


In [57]:
episodios["season"].value_counts().sort_index() #temporada 28 incompleta

,count
season,
1,13
2,22
3,24
4,22
5,22
6,25
7,25
8,25
9,25


### Hallazgos de la exploración

**Episodios**

- 600 episodios de 28 temporadas, emitidos entre el 17/12/1989 y el 16/10/2016. La mayoría de las temporadas tiene entre 20 y 25 episodios; la temporada 1 tiene 13, por ser más corta, y la **temporada 28 tiene solo 4**, (datos se recopilaron poco después del estreno de la 28°temporada).
- No hay filas duplicadas y los `id` van del 1 al 600 sin repetirse.
- Hay pocos nulos: 6 en audiencia (`us_viewers_in_millions`), 4 en `views` y 3 en `imdb_rating` e `imdb_votes`. Son menos del 1% de los episodios.
- `imdb_rating` va de 4,5 a 9,2, con promedio de 7,4 y desvío de 0,73: la mayoría de los episodios se concentra en un rango estrecho.
- La audiencia en EE.UU. va de 2,3 a 33,6 millones de espectadores: hay una diferencia muy grande entre los episodios más y menos vistos, que habrá que analizar en el tiempo.
- `original_air_date` está como texto (`object`), no como fecha, y no hay una columna de año. Para analizar por décadas hay que convertirla y crear esa columna.
- Las filas no están ordenadas cronológicamente (el `head()` arranca en el episodio 10 y el `tail()` termina en el 548).

**Diálogos**

- 158.248 líneas de guion (se descartaron 23 filas mal formadas al cargar), sin filas duplicadas.
- Solo 564 de los 600 episodios tienen líneas de diálogo: 36 episodios no tienen guion y no pueden incluirse en el análisis de protagonismo. [Completar: si se concentran en alguna época.]
- `speaking_line` no es booleana: viene como texto (`"true"` / `"false"`). Hay 132.089 líneas habladas y 26.158 acotaciones, más una fila corrupta con una frase del guion en esa columna.
- `timestamp_in_ms` y `character_id` deberían ser numéricas pero aparecen como texto (`object`), por esa misma fila corrupta.
- Los nulos en `spoken_words`, `normalized_text` y `word_count` (unos 26.000) corresponden a las acotaciones del guion, que no tienen palabras dichas. Los nulos en `character_id` y `raw_character_text` corresponden mayormente a acotaciones sin personaje.
- Las líneas con palabras tienen en promedio casi 10 palabras (mediana 8, máximo 122). Hay líneas con 0 palabras registradas.
- Hay 6.763 nombres de personajes distintos, y un mismo personaje puede aparecer con varios nombres: por ejemplo, 63 nombres contienen "Bart" ("Adult Bart", "Teenage Bart"), aunque "Bart Simpson" concentra la gran mayoría de las líneas. La búsqueda por texto incluye además personajes que no son Bart, como "Bartender", por lo que una unificación automática no sería confiable.

**Para la etapa de transformaciones:**

- Filtrar solo las líneas habladas (`speaking_line == "true"`), lo que también descarta la fila corrupta.
- Convertir `original_air_date` a fecha y crear las columnas de año y década.
- Convertir `timestamp_in_ms` y `character_id` a numéricas.
- Decidir cómo tratar los nulos de rating y audiencia, la temporada 28 (incompleta) y las variantes de nombres de personajes.